In [0]:
%sql
CREATE OR REPLACE TABLE learning.silver.merge_demo (
order_id INT,
amount DOUBLE,
status STRING
)
USING DELTA;

INSERT INTO learning.silver.merge_demo
VALUES (1, 100, 'pending'),
(2, 200, 'pending'),
(3, 300, 'completed');

In [0]:
new_orders = spark.createDataFrame(
[
    (2, 200, 'completed'),
    (3, 400, 'completed'),
    (4, 300, 'pending'),
    (5, 100, 'pending')
],
'order_id: INTEGER, amount: DOUBLE, status: STRING'
)
display(new_orders)

In [0]:
new_orders.createOrReplaceTempView('new_orders')

In [0]:
%sql
MERGE INTO learning.silver.merge_demo AS target
USING new_orders AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN
INSERT *;
SELECT * FROM learning.silver.merge_demo	
Order BY order_id;

In [0]:
%sql
SELECT 
COUNT(*) AS total_orders,
SUM(amount) AS total_amount
FROM learning.silver.merge_demo
WHERE status = 'completed';

In [0]:
%sql
DESCRIBE HISTORY learning.silver.merge_demo

In [0]:
%sql
SELECT * FROM learning.silver.merge_demo VERSION AS OF 4;